In [38]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from datetime import datetime
import pytz

In [39]:
apps = pd.read_csv("Play Store Data.csv")
reviews = pd.read_csv("user_review.csv")

apps.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [40]:
apps['Size'] = apps['Size'].astype(str)
apps['Size'] = apps['Size'].str.replace('M','')
apps['Size'] = apps['Size'].str.replace('k','')
apps['Size'] = apps['Size'].replace('Varies with device', None)
apps['Size'] = pd.to_numeric(apps['Size'], errors='coerce')

apps['Installs'] = apps['Installs'].astype(str)
apps['Installs'] = apps['Installs'].str.replace('+','')
apps['Installs'] = apps['Installs'].str.replace(',','')
apps['Installs'] = pd.to_numeric(apps['Installs'], errors='coerce')

apps['Last Updated'] = pd.to_datetime(apps['Last Updated'], errors='coerce')

apps['Reviews'] = pd.to_numeric(apps['Reviews'], errors='coerce')
apps['Rating'] = pd.to_numeric(apps['Rating'], errors='coerce')

In [41]:
os.makedirs("charts", exist_ok=True)

In [42]:
apps['Rating'] = pd.to_numeric(apps['Rating'], errors='coerce')
apps['Reviews'] = pd.to_numeric(apps['Reviews'], errors='coerce')
apps['Installs'] = pd.to_numeric(apps['Installs'], errors='coerce')

apps = apps.dropna(subset=['Rating','Reviews','Installs'])

In [43]:
ist = pytz.timezone("Asia/Kolkata")
current_hour = datetime.now(ist).hour

In [44]:
if 15 <= current_hour <= 17:

    task1 = apps[(apps['Rating']>=4) &
                 (apps['Size']>=10) &
                 (apps['Last Updated'].dt.month==1)]

    top10 = task1.groupby('Category')['Installs'].sum().nlargest(10)

    task1 = task1[task1['Category'].isin(top10.index)]

    summary = task1.groupby('Category').agg({
        'Rating':'mean',
        'Reviews':'sum'
    }).reset_index()

    fig1 = px.bar(summary,
                  x='Category',
                  y=['Rating','Reviews'],
                  barmode='group',
                  title="Average Rating vs Reviews",
                  color_discrete_sequence=px.colors.qualitative.Bold)

    fig1.write_html("charts/task1.html")
    fig1.show()

else:
    print("Task1 chart visible only between 3PM–5PM IST")

Task1 chart visible only between 3PM–5PM IST


In [45]:
if 18 <= current_hour <= 20:

    task2 = apps.groupby('Category')['Installs'].sum().reset_index()

    task2 = task2[~task2['Category'].str.startswith(('A','C','G','S'))]

    task2 = task2.sort_values('Installs',ascending=False).head(5)

    fig2 = px.bar(task2,
                  x='Category',
                  y='Installs',
                  color='Installs',
                  title="Top 5 Categories by Installs")

    fig2.write_html("charts/task2.html")
    fig2.show()

else:
    print("Task2 chart visible only between 6PM–8PM IST")

In [46]:
if 13 <= current_hour <= 14:

    task3 = apps[(apps['Installs']>10000) &
                 (apps['Revenue']>10000) &
                 (apps['Size']>15) &
                 (apps['Content Rating']=="Everyone")]

    top3 = task3['Category'].value_counts().head(3).index
    task3 = task3[task3['Category'].isin(top3)]

    summary = task3.groupby('Type').agg({
        'Installs':'mean',
        'Revenue':'mean'
    }).reset_index()

    fig3 = go.Figure()

    fig3.add_bar(x=summary['Type'], y=summary['Installs'], name="Avg Installs")
    fig3.add_scatter(x=summary['Type'], y=summary['Revenue'], name="Revenue", yaxis="y2")

    fig3.update_layout(
        title="Free vs Paid Apps",
        yaxis2=dict(overlaying='y', side='right')
    )

    fig3.write_html("charts/task3.html")
    fig3.show()

else:
    print("Task3 chart visible only between 1PM–2PM IST")

Task3 chart visible only between 1PM–2PM IST


In [51]:
task4 = apps[(apps['Reviews']>500) &
             (apps['Category'].str.startswith(('E','C','B'))) &
             (~apps['App'].str.startswith(('X','Y','Z'))) &
             (~apps['App'].str.contains('S'))]

fig4 = px.line(task4,
               x="Last Updated",
               y="Installs",
               color="Category",
               color_discrete_sequence=px.colors.qualitative.Plotly,
               title="Install Trends Over Time")

fig4.write_html("charts/task4.html")
fig4.show()

In [47]:
data = apps.merge(reviews,on="App",how="left")

task5 = data[(data['Rating']>3.5) &
             (data['Reviews']>500) &
             (data['Installs']>50000) &
             (data['Sentiment_Subjectivity']>0.5)]

fig5 = px.scatter(task5,
                  x="Size",
                  y="Rating",
                  size="Installs",
                  color="Category",
                  hover_name="App",
                  color_discrete_sequence=px.colors.qualitative.Dark24,
                  title="App Size vs Rating")

fig5.write_html("charts/task5.html")
fig5.show()

In [48]:
task6 = apps[(apps['Rating']>=4.2) &
             (apps['Reviews']>1000) &
             (apps['Size']>=20) &
             (apps['Size']<=80) &
             (apps['Category'].str.startswith(('T','P'))) &
             (~apps['App'].str.contains(r'\d'))]

fig6 = px.area(task6,
               x="Last Updated",
               y="Installs",
               color="Category",
               color_discrete_sequence=px.colors.qualitative.Set2,
               title="Cumulative Installs")

fig6.write_html("charts/task6.html")
fig6.show()

In [60]:
import webbrowser
import os

html = """
<!DOCTYPE html>
<html>

<head>

<title>Google Play Store Reviews Analytics</title>

<style>

body{
background:#0b0b0b;
font-family:Arial;
color:white;
margin:0;
}

.header{

text-align:center;
padding:25px;
background:#1a1a1a;
font-size:35px;
font-weight:bold;

}

.dashboard{

display:grid;
grid-template-columns: 1fr 1fr;
gap:25px;
padding:30px;

}

.card{

background:#000;
padding:15px;
border-radius:12px;
box-shadow:0 0 15px rgba(0,0,0,0.9);

}

.card h2{

text-align:center;
color:#ffffff;
font-size:20px;
margin-bottom:10px;

}

iframe{

width:100%;
height:420px;
border:none;

}

</style>

</head>

<body>

<div class="header">

<span style="color:#4285F4">G</span>
<span style="color:#EA4335">o</span>
<span style="color:#FBBC05">o</span>
<span style="color:#4285F4">g</span>
<span style="color:#34A853">l</span>
<span style="color:#EA4335">e</span>

Google Play Store Reviews Analytics

</div>

<div class="dashboard">

<div class="card">
<h2>Top Categories</h2>
<iframe src="charts/task1.html"></iframe>
</div>

<div class="card">
<h2>App Type Distribution</h2>
<iframe src="charts/task2.html"></iframe>
</div>

<div class="card">
<h2>Rating Distribution</h2>
<iframe src="charts/task3.html"></iframe>
</div>

<div class="card">
<h2>Install Trends</h2>
<iframe src="charts/task4.html"></iframe>
</div>

<div class="card">
<h2>Reviews Analysis</h2>
<iframe src="charts/task5.html"></iframe>
</div>

<div class="card">
<h2>Sentiment Analysis</h2>
<iframe src="charts/task6.html"></iframe>
</div>

</div>

</body>
</html>
"""

with open("web_page.html","w") as f:
    f.write(html)

path = os.path.abspath("web_page.html")
webbrowser.open("file://" + path)

True